# Hotel Booking & Revenue Analytics

Run this notebook in Google Colab to explore the hotel booking data. At the upload prompt, select the project ZIP bundle (or upload both CSV files individually). The included records are synthetic and illustrative.

In [1]:
%pip -q install pandas plotly

In [ ]:
from google.colab import files
import io
import zipfile
import pandas as pd

print("Upload the project ZIP bundle, or select bookings.csv and room_inventory.csv individually.")
uploaded = files.upload()
required_files = {"bookings.csv", "room_inventory.csv"}
file_contents = dict(uploaded)
zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
if zip_files:
    if len(zip_files) != 1:
        raise ValueError("Upload one project ZIP bundle at a time.")
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_files[0]])) as archive:
        for required_name in required_files:
            matches = [name for name in archive.namelist() if name.rsplit("/", 1)[-1] == required_name]
            if len(matches) > 1:
                raise ValueError(f"ZIP bundle contains more than one {required_name}.")
            if matches:
                file_contents[required_name] = archive.read(matches[0])
missing_files = required_files.difference(file_contents)
if missing_files:
    raise FileNotFoundError(
        "The upload must contain both required CSV files: " + ", ".join(sorted(missing_files))
    )

bookings = pd.read_csv(io.BytesIO(file_contents["bookings.csv"]))
inventory = pd.read_csv(io.BytesIO(file_contents["room_inventory.csv"]))
bookings.columns = bookings.columns.str.strip()
inventory.columns = inventory.columns.str.strip()

if "departure_date" not in bookings.columns:
    if "total_nights" not in bookings.columns:
        raise ValueError("bookings.csv needs departure_date or total_nights.")
    bookings["arrival_date"] = pd.to_datetime(bookings["arrival_date"], errors="raise")
    bookings["departure_date"] = bookings["arrival_date"] + pd.to_timedelta(
        pd.to_numeric(bookings["total_nights"], errors="raise"), unit="D"
    )
else:
    bookings["arrival_date"] = pd.to_datetime(bookings["arrival_date"], errors="raise")
    bookings["departure_date"] = pd.to_datetime(bookings["departure_date"], errors="raise")

if "inventory_date" not in inventory.columns and "date" in inventory.columns:
    inventory = inventory.rename(columns={"date": "inventory_date"})
if "rooms_available" not in inventory.columns and "total_rooms" in inventory.columns:
    inventory = inventory.rename(columns={"total_rooms": "rooms_available"})
if not {"inventory_date", "room_type", "rooms_available"}.issubset(inventory.columns):
    raise ValueError("room_inventory.csv needs date, room_type, and total_rooms (or their accepted aliases).")
inventory["inventory_date"] = pd.to_datetime(inventory["inventory_date"], errors="raise")
bookings["is_canceled"] = (
    bookings["is_canceled"].astype(str).str.strip().str.lower()
    .map({"1": True, "true": True, "yes": True, "canceled": True, "cancelled": True,
          "0": False, "false": False, "no": False, "confirmed": False,
          "not canceled": False, "not cancelled": False})
)
if bookings["is_canceled"].isna().any():
    raise ValueError("bookings.csv contains unsupported is_canceled values.")

print(f"Loaded {len(bookings):,} bookings and {len(inventory):,} inventory rows.")

In [ ]:
from IPython.display import display

for column in ["adr", "rooms_booked", "adults", "children", "babies"]:
    bookings[column] = pd.to_numeric(bookings[column], errors="raise")
confirmed = bookings.loc[~bookings["is_canceled"]].copy()
bookings["total_nights"] = (bookings["departure_date"] - bookings["arrival_date"]).dt.days
confirmed["total_nights"] = (confirmed["departure_date"] - confirmed["arrival_date"]).dt.days
if (bookings["total_nights"] <= 0).any():
    raise ValueError("Every reservation must have a stay longer than zero nights.")

bookings["month"] = bookings["arrival_date"].dt.to_period("M").dt.to_timestamp()
confirmed["month"] = confirmed["arrival_date"].dt.to_period("M").dt.to_timestamp()
confirmed["room_nights"] = confirmed["total_nights"] * confirmed["rooms_booked"]
confirmed["revenue"] = confirmed["adr"] * confirmed["room_nights"]
bookings["status"] = bookings["is_canceled"].map({True: "Canceled", False: "Confirmed"})

# Count each occupied night from check-in up to, but not including, check-out.
stay_rows = []
for row in confirmed[["arrival_date", "departure_date", "room_type", "rooms_booked"]].itertuples(index=False):
    for stay_date in pd.date_range(row.arrival_date, row.departure_date - pd.Timedelta(days=1), freq="D"):
        stay_rows.append((stay_date, row.room_type, float(row.rooms_booked)))
occupied = pd.DataFrame(stay_rows, columns=["date", "room_type", "rooms_occupied"])
if occupied.empty:
    daily_occupied = pd.DataFrame(columns=["date", "rooms_occupied"])
else:
    daily_occupied = occupied.groupby("date", as_index=False)["rooms_occupied"].sum()

inventory["rooms_available"] = pd.to_numeric(inventory["rooms_available"], errors="raise")
inventory_start = bookings["arrival_date"].min()
inventory_end = max(bookings["arrival_date"].max(), confirmed["departure_date"].max() - pd.Timedelta(days=1))
capacity = inventory.loc[inventory["inventory_date"].between(inventory_start, inventory_end)]
capacity = capacity.groupby("inventory_date", as_index=False)["rooms_available"].sum()
capacity = capacity.rename(columns={"inventory_date": "date"})
occupancy = capacity.merge(daily_occupied, on="date", how="left")
occupancy["rooms_occupied"] = occupancy["rooms_occupied"].fillna(0.0)
occupancy["occupancy_rate"] = occupancy["rooms_occupied"] / occupancy["rooms_available"]

room_nights = float(confirmed["room_nights"].sum())
total_revenue = float(confirmed["revenue"].sum())
occupancy_rate = (
    float(occupancy["rooms_occupied"].sum() / occupancy["rooms_available"].sum())
    if not occupancy.empty and occupancy["rooms_available"].sum() > 0 else None
)
kpis = {
    "Total bookings": f"{len(bookings):,}",
    "Confirmed revenue": f"{total_revenue:,.2f}",
    "Average daily rate": f"{total_revenue / room_nights:,.2f}" if room_nights else "0.00",
    "Occupancy rate": f"{occupancy_rate:.1%}" if occupancy_rate is not None else "N/A",
    "Cancellation rate": f"{bookings['is_canceled'].mean():.1%}" if len(bookings) else "0.0%",
    "Confirmed guests": f"{int(confirmed[['adults', 'children', 'babies']].sum().sum()):,}",
}
display(pd.DataFrame(kpis.items(), columns=["Metric", "Value"]))

top_channel = bookings.groupby("booking_channel").size().idxmax()
top_room = confirmed.groupby("room_type")["revenue"].sum().idxmax()
print(f"Top booking channel: {top_channel}")
print(f"Top room type by confirmed revenue: {top_room}")

In [ ]:
import plotly.express as px
from IPython.display import display

template = "plotly_dark"
monthly_bookings = bookings.groupby("month", as_index=False).size().rename(columns={"size": "bookings"})
monthly_revenue = confirmed.groupby("month", as_index=False)["revenue"].sum()
revenue_by_room = confirmed.groupby("room_type", as_index=False)["revenue"].sum().sort_values("revenue")
channel_bookings = bookings.groupby("booking_channel", as_index=False).size().rename(columns={"size": "bookings"}).sort_values("bookings")
monthly_status = bookings.groupby(["month", "status"], as_index=False).size().rename(columns={"size": "bookings"})
monthly_adr = confirmed.groupby("month", as_index=False).agg(revenue=("revenue", "sum"), room_nights=("room_nights", "sum"))
monthly_adr["adr"] = monthly_adr["revenue"] / monthly_adr["room_nights"]
country_guests = confirmed.assign(guests=confirmed[["adults", "children", "babies"]].sum(axis=1)).groupby("country", as_index=False)["guests"].sum().nlargest(10, "guests").sort_values("guests")

figures = [
    px.line(monthly_bookings, x="month", y="bookings", markers=True, title="Monthly bookings", template=template),
    px.bar(monthly_revenue, x="month", y="revenue", title="Monthly confirmed revenue", template=template),
    px.bar(revenue_by_room, x="revenue", y="room_type", orientation="h", title="Revenue by room type", template=template),
    px.bar(channel_bookings, x="bookings", y="booking_channel", orientation="h", title="Bookings by channel", template=template),
    px.bar(monthly_status, x="month", y="bookings", color="status", barmode="stack", title="Monthly cancellations", template=template),
    px.line(monthly_adr, x="month", y="adr", markers=True, title="Average daily rate by month", template=template),
    px.bar(country_guests, x="guests", y="country", orientation="h", title="Guests by country (top 10)", template=template),
    px.line(occupancy, x="date", y="occupancy_rate", title="Daily occupancy", template=template),
]
for figure in figures:
    figure.update_layout(height=380)
    display(figure)